In [1]:
from dotenv import load_dotenv

load_dotenv(override=True)

True

In [2]:
from agents import Agent, Runner, trace

instruction = "You provide help with History problems. Explain your Answer with Reference"
custom_agent = Agent(name="History Tutor", instructions=instruction, model="gpt-4o-mini")

In [3]:
runner = await Runner.run(custom_agent, "Who was the First King of India?")
print(runner.final_output)

OpenAIError: The api_key client option must be set either by passing api_key to the client or by setting the OPENAI_API_KEY environment variable

OPENAI_API_KEY is not set, skipping trace export


In [5]:
from dotenv import load_dotenv
import os
from openai import AsyncOpenAI

load_dotenv(override=True)

# Disable tracing BEFORE importing agents
os.environ["OPENAI_AGENTS_DISABLE_TRACING"] = "1"

from agents import Agent, OpenAIChatCompletionsModel, Runner

# Configure for OpenRouter
# os.environ["OPENAI_API_KEY"] = os.getenv("API_TOKEN")  
# os.environ["OPENAI_BASE_URL"] = "https://openrouter.ai/api/v1"
client = AsyncOpenAI(api_key=os.getenv("API_TOKEN") , base_url="https://openrouter.ai/api/v1")

instruction = "You provide help with History problems. Explain your Answer with Reference"
custom_agent = Agent(
    name="History Tutor", 
    instructions=instruction, 
    model=OpenAIChatCompletionsModel(model="gpt-4o-mini", openai_client=client)

    # model="gpt-4o-mini"
)

# Remove the trace context manager since tracing is disabled
runner = await Runner.run(custom_agent, "Who was the First President of Nigeria?")
print(runner.final_output)

OPENAI_API_KEY is not set, skipping trace export


The first President of Nigeria was Nnamdi Azikiwe. He assumed office on October 1, 1963, when Nigeria became a republic, having previously served as Governor-General after independence from British rule in 1960. Azikiwe was a prominent leader in Nigeria's struggle for independence and was a key figure in the political landscape of the country during the mid-20th century. His presidency marked a significant era in Nigeria’s history, as it set the stage for the political developments that followed, including military coups and civil unrest in the subsequent decades.


OPENAI_API_KEY is not set, skipping trace export


In [2]:
from dotenv import load_dotenv
import os
from openai import OpenAI
from IPython.display import Markdown, display
from pypdf import PdfReader
import gradio as gr

load_dotenv(override=True)

c:\Users\USER\Desktop\code\AI Agents\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


True

In [3]:
# Disable tracing BEFORE importing agents
os.environ["OPENAI_AGENTS_DISABLE_TRACING"] = "1"

from agents import Agent, Runner

# Configure for OpenRouter
os.environ["OPENAI_API_KEY"] = os.getenv("API_TOKEN")
os.environ["OPENAI_BASE_URL"] = "https://openrouter.ai/api/v1"

In [5]:
# --- Read PDF & Summary (same as before) ---
pdfReader = PdfReader("Resources/Profile.pdf")
prof_summary = ""
for page in pdfReader.pages:
    text = page.extract_text()
    if text:
        prof_summary += text + "\n"



In [6]:
prof_summary

"Parag Agrawal - Professional Profile\nSummary\nParag Agrawal is an Indian-American software engineer and entrepreneur, best known for his tenure\nas the CEO of Twitter from November 2021 to October 2022. He was appointed CEO following Jack\nDorsey's resignation but was dismissed after Elon Musk's acquisition of Twitter.\nEducation and Early Career\nAgrawal holds a B.Tech in Computer Science and Engineering from the Indian Institute of\nTechnology (IIT) Bombay. He then pursued a Ph.D. in Computer Science at Stanford University,\nwhere his research focused on uncertainty in data management and integration. Prior to his role at\nTwitter, he held research internships at Microsoft Research and Yahoo! Research.\nCareer at Twitter\nAgrawal joined Twitter in 2011 as a software engineer and quickly rose through the ranks. In 2017,\nhe was appointed Chief Technology Officer (CTO), overseeing Twitter's technical strategy and\nleading initiatives like Project Bluesky, aimed at developing a decent

In [7]:
name = "Parag Agrawal"

system_prompt = (
    f"You are acting as {name}, representing {name} on their website. "
    f"Your role is to answer questions specifically about {name}'s career, background, skills, and experience. "
    f"You must faithfully and accurately portray {name} in all interactions. "
    f"You have access to a detailed summary of {name}'s background and their LinkedIn profile, which you should use to inform your answers. "
    f"Maintain a professional, engaging, and approachable tone, as if you are speaking to a potential client or future employer visiting the site. "
    f"If you are unsure of an answer, it is better to honestly acknowledge that than to guess."
    f" LinkedIn Profile:\n{prof_summary}\n\n"
    f"Using this context, please converse naturally and consistently, always staying in character as {name}."
)

In [11]:
custom_agent = Agent(
    name="Portfolio Chatbot",
    instructions=system_prompt,
    model="gpt-4o-mini"
)

In [12]:
async def liveChat(message, history):
    # Build conversation history in OpenAI format
    messages = [{"role": msg["role"], "content": msg["content"]} for msg in history]
    messages.append({"role": "user", "content": message})

    result = await Runner.run(custom_agent, messages)
    return result.final_output

In [13]:
gr.ChatInterface(
    liveChat,
    title="Chat with Parag Agrawal",
    description="Ask me anything about my career, background, skills, and experience!",
).launch()

* Running on local URL:  http://127.0.0.1:7862
* To create a public link, set `share=True` in `launch()`.


In [ ]:
from dotenv import load_dotenv
import os

load_dotenv(override=True)

# Disable tracing BEFORE importing agents
os.environ["OPENAI_AGENTS_DISABLE_TRACING"] = "1"

# Configure for OpenRouter
os.environ["OPENAI_API_KEY"] = os.getenv("API_TOKEN")
os.environ["OPENAI_BASE_URL"] = "https://openrouter.ai/api/v1"

from agents import Agent, Runner, function_tool

# Define a tool using the @tool decorator
@function_tool
def get_weather(city: str) -> str:
    """
    Get the current weather for a city.
    
    Args:
        city: The name of the city to get weather for
    """
    weather_data = {
        "london": "Cloudy, 15°C",
        "tokyo": "Sunny, 22°C",
        "new york": "Rainy, 18°C"
    }
    return weather_data.get(city.lower(), f"Weather data not available for {city}")

# Create agent with the tool
weather_agent = Agent(
    name="WeatherBot",
    instructions="You help users check the weather. Use the get_weather tool when asked about weather.",
    model="gpt-4o-mini",
    tools=[get_weather]
)

# Run it
runner = await Runner.run(weather_agent, "What's the weather in Tokyo?")
print(runner.final_output)

The weather in Tokyo is sunny with a temperature of 22°C.


In [4]:
@function_tool
def get_user_id(email: str) -> str:
    """Look up user ID from email."""
    return "user_123"

@function_tool
def get_user_orders(user_id: str) -> str:
    """Get orders for a user ID."""
    return "Order #1: Laptop, Order #2: Mouse"

@function_tool
def get_order_status(order_id: str) -> str:
    """Get status of an order."""
    return "Shipped - arriving tomorrow"

agent = Agent(
    name="OrderAssistant",
    instructions="Help users check their order status. First find their user ID, then their orders, then status.",
    tools=[get_user_id, get_user_orders, get_order_status]
)

runner = await Runner.run(agent, "What's the status of my orders? My email is john@example.com")
print(runner.final_output)
# User asks: "What's the status of my orders? My email is john@example.com"
# Agent will:
# 1. Call get_user_id("john@example.com") → "user_123"
# 2. Call get_user_orders("user_123") → "Order #1: Laptop..."
# 3. Call get_order_status("1") → "Shipped..."
# 4. Respond with compiled information

Error getting response: Error code: 402 - {'error': {'message': 'This request requires more credits, or fewer max_tokens. You requested up to 65536 tokens, but can only afford 3986. To increase, visit https://openrouter.ai/settings/credits and upgrade to a paid account', 'code': 402, 'metadata': {'provider_name': None}}, 'user_id': 'user_36C4aRh1DiqT1kuAEJVlSveiukv'}. (request_id: None)


APIStatusError: Error code: 402 - {'error': {'message': 'This request requires more credits, or fewer max_tokens. You requested up to 65536 tokens, but can only afford 3986. To increase, visit https://openrouter.ai/settings/credits and upgrade to a paid account', 'code': 402, 'metadata': {'provider_name': None}}, 'user_id': 'user_36C4aRh1DiqT1kuAEJVlSveiukv'}

In [5]:
@function_tool
def get_weather(city: str) -> str:
    """Get weather for a city."""
    return f"{city}: Sunny, 25°C"

@function_tool
def get_news(topic: str) -> str:
    """Get latest news on a topic."""
    return f"Latest {topic} news: ..."

@function_tool
def get_stock(symbol: str) -> str:
    """Get stock price."""
    return f"{symbol}: $150.00"

agent = Agent(
    name="MorningBriefing",
    instructions="Provide a morning briefing with weather, news, and stock info.",
    tools=[get_weather, get_news, get_stock]
)

runner = await Runner.run(agent, "Give me my morning briefing for NYC, tech news, and AAPL")
print(runner.final_output)

# User: "Give me my morning briefing for NYC, tech news, and AAPL"
# Agent calls all 3 tools in parallel → Compiles response

Error getting response: Error code: 402 - {'error': {'message': 'This request requires more credits, or fewer max_tokens. You requested up to 65536 tokens, but can only afford 3986. To increase, visit https://openrouter.ai/settings/credits and upgrade to a paid account', 'code': 402, 'metadata': {'provider_name': None}}, 'user_id': 'user_36C4aRh1DiqT1kuAEJVlSveiukv'}. (request_id: None)


APIStatusError: Error code: 402 - {'error': {'message': 'This request requires more credits, or fewer max_tokens. You requested up to 65536 tokens, but can only afford 3986. To increase, visit https://openrouter.ai/settings/credits and upgrade to a paid account', 'code': 402, 'metadata': {'provider_name': None}}, 'user_id': 'user_36C4aRh1DiqT1kuAEJVlSveiukv'}

In [6]:
@function_tool
def search_web(query: str) -> str:
    """Search the web for current information."""
    return f"Web results for '{query}': ..."

@function_tool
def search_database(query: str) -> str:
    """Search internal company database."""
    return f"Database results for '{query}': ..."

@function_tool
def search_documents(query: str) -> str:
    """Search uploaded documents."""
    return f"Document results for '{query}': ..."

agent = Agent(
    name="SmartSearch",
    instructions="""You are a search assistant.
    
    Choose the right search based on the query:
    - For current events/general info → use search_web
    - For company/employee info → use search_database
    - For policy/procedure questions → use search_documents
    """,
    tools=[search_web, search_database, search_documents]
)

runner = await Runner.run(agent, "Find the latest company policy on remote work.")
print(runner.final_output)

Error getting response: Error code: 402 - {'error': {'message': 'This request requires more credits, or fewer max_tokens. You requested up to 65536 tokens, but can only afford 3986. To increase, visit https://openrouter.ai/settings/credits and upgrade to a paid account', 'code': 402, 'metadata': {'provider_name': None}}, 'user_id': 'user_36C4aRh1DiqT1kuAEJVlSveiukv'}. (request_id: None)


APIStatusError: Error code: 402 - {'error': {'message': 'This request requires more credits, or fewer max_tokens. You requested up to 65536 tokens, but can only afford 3986. To increase, visit https://openrouter.ai/settings/credits and upgrade to a paid account', 'code': 402, 'metadata': {'provider_name': None}}, 'user_id': 'user_36C4aRh1DiqT1kuAEJVlSveiukv'}

In [ ]:
from dotenv import load_dotenv
import os

load_dotenv(override=True)

os.environ["OPENAI_AGENTS_DISABLE_TRACING"] = "1"
os.environ["OPENAI_API_KEY"] = os.getenv("API_TOKEN")
os.environ["OPENAI_BASE_URL"] = "https://openrouter.ai/api/v1"

from agents import Agent, Runner, function_tool
import requests

@function_tool
def get_weather(city: str) -> str:
    """
    Get the current weather for a city.
    
    Args:
        city: The name of the city to get weather for
    """
    try:
        url = f"https://wttr.in/{city}?format=%C+%t+%h+%w"
        response = requests.get(url, timeout=10)
        if response.status_code == 200:
            return f"Weather in {city}: {response.text}"
        return f"Could not fetch weather for {city}"
    except Exception as e:
        return f"Error: {str(e)}"

@function_tool
def search_wikipedia(query: str) -> str:
    """
    Search Wikipedia for information.
    
    Args:
        query: The topic to search for
    """
    try:
        url = f"https://en.wikipedia.org/api/rest_v1/page/summary/{query}"
        response = requests.get(url, timeout=10)
        if response.status_code == 200:
            data = response.json()
            return data.get("extract", "No information found")
        return f"No Wikipedia article found for {query}"
    except Exception as e:
        return f"Error: {str(e)}"

@function_tool
def get_random_joke() -> str:
    """Get a random joke."""
    try:
        response = requests.get(
            "https://official-joke-api.appspot.com/random_joke",
            timeout=10
        )
        if response.status_code == 200:
            data = response.json()
            return f"{data['setup']} - {data['punchline']}"
        return "Could not fetch joke"
    except Exception as e:
        return f"Error: {str(e)}"


assistant = Agent(
    name="RealWorldAssistant",
    instructions="""You are a helpful assistant with real-world capabilities.

You can:
- Get LIVE weather for any city using get_weather
- Search Wikipedia for information using search_wikipedia  
- Tell jokes using get_random_joke

Always use your tools when asked about weather, facts, or jokes.
Be friendly and informative.""",
    model="gpt-4o-mini",
    tools=[get_weather, search_wikipedia, get_random_joke]
)


if __name__ == "__main__":
    print("=== Weather Test ===")
    runner = await Runner.run(assistant, "What's the weather in Lagos, Nigeria?")
    print(runner.final_output)
    
    print("\n=== Wikipedia Test ===")
    runner = await Runner.run(assistant, "Tell me about Python programming language")
    print(runner.final_output)
    
    print("\n=== Joke Test ===")
    runner = await Runner.run(assistant, "Tell me a joke")
    print(runner.final_output)

=== Weather Test ===
The current weather in Lagos, Nigeria is foggy with a temperature of 8°C. The humidity is at 99%, and there's a light breeze blowing at 5 km/h.

=== Wikipedia Test ===
It seems I'm unable to find a specific Wikipedia article for the Python programming language at the moment. However, I can provide you with some general information about it!

**Python** is a high-level, interpreted programming language known for its readability and simplicity. It was created by Guido van Rossum and first released in 1991. Python's design philosophy emphasizes code readability with its use of significant indentation. 

### Key Features of Python:
- **Easy to Learn and Use**: Python has a simple syntax that mimics natural language, making it accessible for beginners.
- **Interpreted Language**: Python code is executed line by line, which makes debugging easier.
- **Dynamically Typed**: You don't need to define the data type of a variable when you declare it. Python determines the type

In [12]:
from dotenv import load_dotenv
import os

load_dotenv(override=True)

os.environ["OPENAI_AGENTS_DISABLE_TRACING"] = "1"
os.environ["OPENAI_API_KEY"] = os.getenv("API_TOKEN")
os.environ["OPENAI_BASE_URL"] = "https://openrouter.ai/api/v1"

from agents import Agent, Runner, function_tool
import requests

@function_tool
def get_weather(city: str) -> str:
    """
    Get the current weather for a city.
    
    Args:
        city: The name of the city to get weather for
    """
    try:
        url = f"https://wttr.in/{city}?format=%C+%t+%h+%w"
        response = requests.get(url, timeout=10)
        if response.status_code == 200:
            return f"Weather in {city}: {response.text}"
        return f"Could not fetch weather for {city}"
    except Exception as e:
        return f"Error: {str(e)}"

@function_tool
def search_wikipedia(query: str) -> str:
    """
    Search Wikipedia for information.
    
    Args:
        query: The topic to search for
    """
    try:
        url = f"https://en.wikipedia.org/api/rest_v1/page/summary/{query}"
        response = requests.get(url, timeout=10)
        if response.status_code == 200:
            data = response.json()
            return data.get("extract", "No information found")
        return f"No Wikipedia article found for {query}"
    except Exception as e:
        return f"Error: {str(e)}"

@function_tool
def get_random_joke() -> str:
    """Get a random joke."""
    try:
        response = requests.get(
            "https://official-joke-api.appspot.com/random_joke",
            timeout=10
        )
        if response.status_code == 200:
            data = response.json()
            return f"{data['setup']} - {data['punchline']}"
        return "Could not fetch joke"
    except Exception as e:
        return f"Error: {str(e)}"


assistant = Agent(
    name="RealWorldAssistant",
    instructions="""You are a helpful assistant with real-world capabilities.

You can:
- Get LIVE weather for any city using get_weather
- Search Wikipedia for information using search_wikipedia  
- Tell jokes using get_random_joke

Always use your tools when asked about weather, facts, or jokes.
Be friendly and informative.""",
    model="gpt-4o-mini",
    tools=[get_weather, search_wikipedia, get_random_joke]
)


import asyncio

async def main():
    print("=== Weather Test ===")
    runner = await Runner.run(assistant, "What's the weather in Lagos, Nigeria?")
    print(runner.final_output)

    print("\n=== Wikipedia Test ===")
    runner = await Runner.run(assistant, "Tell me about Python programming language")
    print(runner.final_output)

    print("\n=== Joke Test ===")
    runner = await Runner.run(assistant, "Tell me a joke")
    print(runner.final_output)

if __name__ == "__main__":
    asyncio.run(main())

RuntimeError: asyncio.run() cannot be called from a running event loop

In [ ]:
from agents import Agent, Runner, handoff

# Specialist agents
billing_agent = Agent(
    name="BillingSpecialist",
    instructions="""You handle billing questions:
    - Payment issues
    - Refunds
    - Invoice requests
    Be helpful and resolve issues quickly."""
)

technical_agent = Agent(
    name="TechnicalSupport",
    instructions="""You handle technical issues:
    - Bug reports
    - How-to questions
    - Feature requests
    Ask clarifying questions to understand the issue."""
)

# Triage agent that routes to specialists
triage_agent = Agent(
    name="Triage",
    instructions="""You are the first point of contact.
    
    Route customers to the right specialist:
    - Billing/payment issues → transfer to BillingSpecialist
    - Technical problems → transfer to TechnicalSupport
    
    Ask one clarifying question if needed before routing.
    """,
    handoffs=[
        handoff(billing_agent),
        handoff(technical_agent)
    ]
)

# Run the triage agent
result = await Runner.run(triage_agent, "I was charged twice for my subscription!")
# → Will handoff to billing_agent

Error getting response: Error code: 402 - {'error': {'message': 'This request requires more credits, or fewer max_tokens. You requested up to 65536 tokens, but can only afford 3974. To increase, visit https://openrouter.ai/settings/credits and upgrade to a paid account', 'code': 402, 'metadata': {'provider_name': None}}, 'user_id': 'user_36C4aRh1DiqT1kuAEJVlSveiukv'}. (request_id: None)


APIStatusError: Error code: 402 - {'error': {'message': 'This request requires more credits, or fewer max_tokens. You requested up to 65536 tokens, but can only afford 3974. To increase, visit https://openrouter.ai/settings/credits and upgrade to a paid account', 'code': 402, 'metadata': {'provider_name': None}}, 'user_id': 'user_36C4aRh1DiqT1kuAEJVlSveiukv'}

In [ ]:
from dotenv import load_dotenv
import os

load_dotenv(override=True)

os.environ["OPENAI_AGENTS_DISABLE_TRACING"] = "1"
os.environ["OPENAI_API_KEY"] = os.getenv("API_TOKEN")
os.environ["OPENAI_BASE_URL"] = "https://openrouter.ai/api/v1"

from agents import Agent, Runner, function_tool

# Create specialist agents
researcher = Agent(
    name="Researcher",
    instructions="You research topics thoroughly and return detailed findings.",
    model="gpt-4o"
)

writer = Agent(
    name="Writer", 
    instructions="You write clear, engaging content based on provided information.",
    model="gpt-4o"
)

# tool functions that use agents
@function_tool
async def research_topic(topic: str) -> str:
    """
    Research a topic thoroughly.
    
    Args:
        topic: The topic to research
    """
    result = await Runner.run(researcher, f"Research this topic: {topic}")
    return result.final_output

@function_tool
async def write_content(brief: str) -> str:
    """
    Write content based on a brief.
    
    Args:
        brief: The writing brief with topic and key points
    """
    result = await Runner.run(writer, brief)
    return result.final_output

# Orchestrator that uses other agents as tools
orchestrator = Agent(
    name="ContentManager",
    instructions="""You manage content creation.
    
    When asked to create content:
    1. Use research_topic to gather information
    2. Use write_content to create the final piece
    3. Review and present the result
    """,
    model="gpt-4o",
    tools=[research_topic, write_content]
)

# Usage
runner = await Runner.run(
    orchestrator,
    "Create a blog post about the benefits of meditation"
)
print(runner.final_output)

Error getting response: Error code: 402 - {'error': {'message': 'This request requires more credits, or fewer max_tokens. You requested up to 16384 tokens, but can only afford 3179. To increase, visit https://openrouter.ai/settings/credits and upgrade to a paid account', 'code': 402, 'metadata': {'provider_name': None}}, 'user_id': 'user_36C4aRh1DiqT1kuAEJVlSveiukv'}. (request_id: None)


APIStatusError: Error code: 402 - {'error': {'message': 'This request requires more credits, or fewer max_tokens. You requested up to 16384 tokens, but can only afford 3179. To increase, visit https://openrouter.ai/settings/credits and upgrade to a paid account', 'code': 402, 'metadata': {'provider_name': None}}, 'user_id': 'user_36C4aRh1DiqT1kuAEJVlSveiukv'}

C:\Program Files\WindowsApps\PythonSoftwareFoundation.Python.3.13_3.13.2544.0_x64__qbz5n2kfra8p0\Lib\re\__init__.py:310: RuntimeWarning: coroutine 'main' was never awaited
  return pattern.translate(_special_chars_map)


In [1]:
from dotenv import load_dotenv
import os

load_dotenv(override=True)

# Disable default OpenAI tracing
# os.environ["OPENAI_AGENTS_DISABLE_TRACING"] = "1"

from agents import Agent, Runner, set_trace_processors
from agents.tracing import TracingProcessor

# Custom local logger with all required methods
class LocalLogger(TracingProcessor):
    def on_trace_start(self, trace):
        print(f"Trace started: {trace.trace_id}")
    
    def on_trace_end(self, trace):
        print(f"Trace ended: {trace.trace_id}")
    
    def on_span_start(self, span):
        print(f"Span started: {span.span_id}")
    
    def on_span_end(self, span):
        print(f"Span ended: {span.span_id} - {span.span_data}")
    
    def force_flush(self):
        pass
    
    def shutdown(self):
        pass

# Set custom processor
set_trace_processors([LocalLogger()])

# Configure for OpenRouter
os.environ["OPENAI_API_KEY"] = os.getenv("API_TOKEN")  
os.environ["OPENAI_BASE_URL"] = "https://openrouter.ai/api/v1"

instruction = "You provide help with History problems. Explain your Answer with Reference"
custom_agent = Agent(
    name="History Tutor", 
    instructions=instruction, 
    model="gpt-4o-mini"
)

runner = await Runner.run(custom_agent, "Who was the First President of USA?")
print(runner.final_output)

Trace started: trace_70a906da98704b48bd973af4c5d54082
Span started: span_a280bf3fd58343e199374a7a
Span started: span_56b8eb4032414b66b9d79457
Span ended: span_56b8eb4032414b66b9d79457 - <agents.tracing.span_data.ResponseSpanData object at 0x00000260A7739590>
Span ended: span_a280bf3fd58343e199374a7a - <agents.tracing.span_data.AgentSpanData object at 0x00000260A4F8AC60>
Trace ended: trace_70a906da98704b48bd973af4c5d54082
The first President of the United States was George Washington. He served from April 30, 1789, to March 4, 1797. Washington is often referred to as the "Father of His Country" for his role in leading the nation during its formative years.


In [1]:
from dotenv import load_dotenv
import os
import json
from datetime import datetime

load_dotenv(override=True)

# Disable default OpenAI tracing
os.environ["OPENAI_AGENTS_DISABLE_TRACING"] = "1"

from agents import Agent, Runner, set_trace_processors
from agents.tracing import TracingProcessor

# Custom local logger that saves to file
class LocalLogger(TracingProcessor):
    def __init__(self, filename="traces.json"):
        self.filename = filename
        self.traces = []
    
    def _write(self, event_type, data):
        entry = {
            "timestamp": datetime.now().isoformat(),
            "event": event_type,
            "data": str(data)
        }
        with open(self.filename, "a") as f:
            f.write(json.dumps(entry) + "\n")
    
    def on_trace_start(self, trace):
        self._write("trace_start", trace.trace_id)
    
    def on_trace_end(self, trace):
        self._write("trace_end", trace.trace_id)
    
    def on_span_start(self, span):
        self._write("span_start", span.span_id)
    
    def on_span_end(self, span):
        self._write("span_end", {"span_id": span.span_id, "data": str(span.span_data)})
    
    def force_flush(self):
        pass
    
    def shutdown(self):
        pass

# Set custom processor
set_trace_processors([LocalLogger("my_traces.json")])

# Configure for OpenRouter
os.environ["OPENAI_API_KEY"] = os.getenv("API_TOKEN")  
os.environ["OPENAI_BASE_URL"] = "https://openrouter.ai/api/v1"

instruction = "You provide help with History problems. Explain your Answer with Reference"
custom_agent = Agent(
    name="History Tutor", 
    instructions=instruction, 
    model="gpt-4o-mini"
)

runner = await Runner.run(custom_agent, "Who was the First President of USA?")
print(runner.final_output)

CancelledError: 

In [ ]:
from openai import OpenAI
import os

client = OpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key=os.getenv("API_TOKEN")
)

response = client.chat.completions.create(
    model="deepseek/deepseek-chat",
    messages=[
        {"role": "system", "content": "You provide help with History problems. Explain your Answer with Reference"},
        {"role": "user", "content": "Who was the First King of India?"}
    ]
)

print(response.choices[0].message.content)

In [2]:
from dotenv import load_dotenv
import os
from winotify import Notification, audio
import threading
from datetime import datetime, timedelta
import re

# toaster = ToastNotifier()

load_dotenv(override=True)

os.environ["OPENAI_AGENTS_DISABLE_TRACING"] = "1"
os.environ["OPENAI_API_KEY"] = os.getenv("API_TOKEN")
os.environ["OPENAI_BASE_URL"] = "https://openrouter.ai/api/v1"

In [3]:
from agents import Agent, Runner, function_tool
from datetime import datetime

# In-memory storage (use a database in production)
todos = []
reminders = []

In [4]:
@function_tool
def add_todo(task: str, priority: str = "medium") -> str:
    """
    Add a task to the todo list.
    
    Args:
        task: The task description
        priority: Priority level (low, medium, high)
    """
    todo = {"task": task, "priority": priority, "done": False, "id": len(todos) + 1}
    todos.append(todo)
    return f"Added: '{task}' with {priority} priority (ID: {todo['id']})"

@function_tool
def list_todos() -> str:
    """List all todos."""
    if not todos:
        return "No todos yet!"
    return "\n".join([
        f"[{'✓' if t['done'] else ' '}] {t['id']}. {t['task']} ({t['priority']})"
        for t in todos
    ])

@function_tool
def complete_todo(todo_id: int) -> str:
    """
    Mark a todo as complete.
    
    Args:
        todo_id: The ID of the todo to complete
    """
    for todo in todos:
        if todo['id'] == todo_id:
            todo['done'] = True
            return f"Completed: '{todo['task']}'"
    return f"Todo {todo_id} not found"


def parse_time(time_str: str) -> datetime:
    """Parse time string like 'in 5 minutes', 'in 1 hour', 'at 3pm'"""
    now = datetime.now()
    time_str = time_str.lower().strip()
    
    # Handle "in X minutes/hours"
    if "in" in time_str:
        match = re.search(r'in\s+(\d+)\s*(minute|min|hour|hr|second|sec)', time_str)
        if match:
            amount = int(match.group(1))
            unit = match.group(2)
            if 'min' in unit:
                return now + timedelta(minutes=amount)
            elif 'hour' in unit or 'hr' in unit:
                return now + timedelta(hours=amount)
            elif 'sec' in unit:
                return now + timedelta(seconds=amount)
    
    # Handle "at 3pm", "at 15:00"
    if "at" in time_str:
        match = re.search(r'at\s+(\d{1,2})(?::(\d{2}))?\s*(am|pm)?', time_str)
        if match:
            hour = int(match.group(1))
            minute = int(match.group(2)) if match.group(2) else 0
            period = match.group(3)
            
            if period == 'pm' and hour != 12:
                hour += 12
            elif period == 'am' and hour == 12:
                hour = 0
            
            target = now.replace(hour=hour, minute=minute, second=0)
            if target <= now:
                target += timedelta(days=1)
            return target
    
    # Default: 1 minute from now
    return now + timedelta(minutes=1)

# Replace the send_notification function
def send_notification(message: str, delay_seconds: float):
    """Send notification after delay"""
    def notify():
        threading.Event().wait(delay_seconds)
        toast = Notification(
            app_id="Personal Assistant",
            title="⏰ Reminder",
            msg=message,
            duration="short"
        )
        toast.set_audio(audio.Default, loop=False)
        toast.show()
    
    thread = threading.Thread(target=notify)
    thread.daemon = True
    thread.start()

@function_tool
def set_reminder(message: str, time: str) -> str:
    """
    Set a reminder that sends a Windows notification.
    
    Args:
        message: What to be reminded about
        time: When to be reminded (e.g., "in 5 minutes", "in 1 hour", "at 3pm")
    """
    target_time = parse_time(time)
    delay = (target_time - datetime.now()).total_seconds()
    
    if delay < 0:
        delay = 0
    
    reminder = {
        "message": message, 
        "time": time, 
        "target": target_time.isoformat(),
        "created": datetime.now().isoformat()
    }
    reminders.append(reminder)
    
    # Schedule the notification
    send_notification(message, delay)
    
    return f"✅ Reminder set: '{message}' for {time} (will notify at {target_time.strftime('%H:%M:%S')})"


@function_tool
def get_current_time() -> str:
    """Get the current date and time."""
    return datetime.now().strftime("%Y-%m-%d %H:%M:%S")

In [5]:
# Create the assistant
assistant = Agent(
    name="PersonalAssistant",
    instructions="""You are a helpful personal assistant.
    
    You can help with:
    - Managing todos (add, list, complete)
    - Setting reminders
    - Telling the time
    
    Be friendly and proactive. If the user adds a todo, 
    ask if they want to set a reminder for it.
    """,
    model="gpt-4o-mini",
    tools=[add_todo, list_todos, complete_todo, set_reminder, get_current_time]
)

In [ ]:
async def chat():
    print("Personal Assistant ready! Type 'quit' to exit.\n")
    
    while True:
        user_input = input("You: ")
        if user_input.lower() == 'quit':
            break
        
        runner = await Runner.run(assistant, user_input)
        print(f"Assistant: {runner.final_output}\n")

await chat()

Personal Assistant ready! Type 'quit' to exit.

Assistant: The current time is 4:15 PM. How can I assist you further?

Assistant: I've set a reminder for you to pray at 4:17 PM. If you need help with anything else, just let me know!

Assistant: The current time is 4:20 PM. If you need anything else, feel free to ask!

Assistant: I've set a reminder for you to pray at 4:22 PM. If you need help with anything else, just let me know!

Assistant: I've added "buy groceries" to your todo list! Would you like to set a reminder for it?

Assistant: You have one todo on your list:

1. Buy groceries (medium priority)

Would you like to set a reminder for this task?

Assistant: You have the following todo:

1. Buy groceries (medium)

Would you like to set a reminder for it?

